In [ ]:
# =========================
# ГЛОБАЛЬНЫЕ НАСТРОЙКИ
# =========================

PUBLIC_KEY = "https://disk.yandex.ru/d/HOZYbvc_LG0viA"
TARGET_FOLDER_PATH = "/2_Artefacts_reduction_Low_Res_High_Res/LowStat_HighStat"
DOWNLOAD_DIR = "filestore/filestorage" # Папка появится прямо рядом с ноутбуком

PAGE_SIZE = 100
OVERWRITE = False
CHUNK_SIZE = 1024 * 1024  # 1 MB

# =========================
# ЗАВИСИМОСТИ
# =========================
import os
from pathlib import Path
import requests
import urllib3
from urllib.parse import unquote
from tqdm.auto import tqdm

# Отключаем предупреждения о несекретных соединениях
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =========================
# YANDEX DISK API
# =========================
API_RESOURCES = "https://cloud-api.yandex.net/v1/disk/public/resources"
API_DOWNLOAD = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

def get_resources(public_key, path="", offset=0):
    params = {
        "public_key": public_key,
        "path": path,
        "limit": PAGE_SIZE,
        "offset": offset
    }
    response = requests.get(API_RESOURCES, params=params, verify=False)
    response.raise_for_status()
    return response.json()

def get_download_url(public_key, path):
    params = {
        "public_key": public_key,
        "path": path,
    }
    response = requests.get(API_DOWNLOAD, params=params, verify=False)
    response.raise_for_status()
    return response.json()["href"]

def download_file(download_url, local_path):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists() and not OVERWRITE:
        return

    with requests.get(download_url, stream=True, verify=False) as response:
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))

        with open(local_path, "wb") as f:
            with tqdm(
                total=total_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc=local_path.name,
                leave=False, 
            ) as progress:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        f.write(chunk)
                        progress.update(len(chunk))

def process_folder(public_url, remote_path, local_root=Path(DOWNLOAD_DIR)):
    offset = 0
    
    while True:
        data = get_resources(public_url, remote_path, offset)
        embedded = data.get("_embedded", {})
        items = embedded.get("items", [])

        if not items:
            break

        for item in items:
            item_type = item.get("type")
            
            # Декодируем путь (на случай, если API вернет кракозябры вроде %20)
            item_path = unquote(item["path"])

            # МАГИЯ ЗДЕСЬ: Отрезаем весь мусор до нужных нам папок
            split_keyword = "LowStat_HighStat/"
            if split_keyword in item_path:
                # Берем всё, что идет ПОСЛЕ "LowStat_HighStat/"
                relative_path = item_path.split(split_keyword, 1)[-1]
            else:
                # Защита от ошибок: если вдруг путь странный, берем только имя файла/папки
                relative_path = Path(item_path).name

            local_path = local_root / relative_path

            if item_type == "dir":
                print(f"📁 Найдена папка: {relative_path} (Сбор файлов...)")
                # В API Яндекса отправляем оригинальный путь из словаря
                next_remote_path = item["path"].replace("disk:", "")
                process_folder(public_url, remote_path=next_remote_path, local_root=local_root)

            elif item_type == "file":
                download_url = item.get("file")
                if not download_url:
                    download_url = get_download_url(public_url, item["path"].replace("disk:", ""))
                download_file(download_url, local_path)

        offset += PAGE_SIZE
        total_items = embedded.get("total", 0)
        if offset >= total_items:
            break

# =========================
# ЗАПУСК
# =========================
if __name__ == "__main__":
    Path(DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)

    print("🚀 Начинаю скачивание датасета...\n")
    process_folder(PUBLIC_KEY, TARGET_FOLDER_PATH)
    print(f"\n🎉 Загрузка завершена! Данные лежат в {DOWNLOAD_DIR}")

🚀 Начинаю скачивание датасета...

📁 Найдена папка: V_beton30_angle005 (Сбор файлов...)


📁 Найдена папка: V_beton30_angle05 (Сбор файлов...)



🎉 Загрузка завершена! Данные лежат в filestore/filestorage
